# q client for DB Service
The DB Service q client provides a thin wrapper over the DB Service APIs, making it easier to connect to a running service and perform client operations from q.

Use this client to create a session, run queries, manage tables, and interact with DB Service from q.

## Requirements
- kdb-x
- A running DB Service instance

## Initialize Runtime

In [1]:
# Load PyKX for q notebook cells
import pykx as kx


Welcome to KDB-X Community Edition!
For Community support, please visit https://kx.com/slack
Tutorials can be found at https://github.com/KxSystems/tutorials
Ready to go beyond the Community Edition? Email preview@kx.com



## Load the client
From the `dbservice-qclient` repository root, load the client:

In [2]:
%%q
dbs:use`kx.dbservice_client
session:dbs.createSession[]

## Endpoint Configuration
The client uses `http://localhost:8080` as the default base URL.

Create a session with the default base URL: `session:dbs.createSession[]`

Or set the base URL explicitly: `session:dbs.createSession["localhost:8080"]`

## Managing Tables
Use these calls to define and inspect table schemas in DB Service.

In [3]:
%%q
// List tables (empty to begin with)
session.listTables[]

()


In [4]:
%%q
// Define columns
fxquoteCols:(`name`type!("trddate";"date");`name`type!("ts";"timestamp");`name`type`attrMem`attrDisk`attrOrd!("sym";"symbol";"grouped";"parted";"parted");`name`type!("bid";"float");`name`type!("ask";"float"))

// Create partitioned table ('fxquote')
session.createTable[`table`type`prtnCol`sortColsDisk`sortColsOrd`columns!("fxquote";"partitioned";"ts";enlist "sym";enlist "sym";fxquoteCols)]

name    | "2d72740d-8f09-9ced-0c36-923c602eeb7d"
pipeline| ""
database| "db"
updtype | "schemaChange"
status  | "completed"
details | ()
tbls    | ()
dates   | ()
progress| `cmdCurrent`cmdIndex`cmdTotal`subCurrent`subIndex`subTotal!("";0f;0..
error   | ""
warnings| ()
updated | "2026-04-27T12:22:38.946360125"


In [ ]:
%%q
// List tables ('fxquote' table returned)
session.listTables[]

"fxquote"


In [6]:
%%q
// Describe the 'fxquote' table
session.describeTable["fxquote"]

type        | "partitioned"
prtnCol     | "ts"
sortColsDisk| ,"sym"
sortColsOrd | ,"sym"
columns     | (`name`type!("trddate";"date");`name`type!("ts";"timestamp");`n..
name        | "fxquote"


## Importing Data
DB Service supports both `file-based` and `in-memory` ingest.
Any file you want to import must first be copied into the DB Service `imports` staging directory, for example: `~/.kx/db-service/data/imports/`

### Import CSV

In [7]:
%%q
// Import a CSV file into the existing 'fxquote' table
job:session.importFiles[`table`path!("fxquote";"fxquote.csv.gz")]

In [8]:
%%q
// Check the status of the above import job
session.getImport[job`name]

name    | "93535245-e957-3b0d-c445-4ec4657d746b"
pipeline| ""
database| "db"
updtype | "ingest"
status  | "completed"
details | `preprocessDone`subsessions`dates`tables!("2026-04-27T12:22:58.2306..
tbls    | ,"fxquote"
dates   | ,"2026-03-02"
progress| `cmdCurrent`cmdIndex`cmdTotal`subCurrent`subIndex`subTotal!("";1f;1..
error   | ""
warnings| ()
updated | "2026-04-27T12:22:58.519696558"


In [9]:
%%q 
// Import a parquet file into the existing 'fxquote' table
job:session.importFiles[`table`path!("fxquote";"fxquote.parquet")]

In [10]:
%%q
// Import a CSV file and create the 'instruments' table automatically if it does not exist
job:session.importFiles[`table`path`createTable!("instruments";"instruments.csv";1b)]

### Import JSON
Import rows directly from q without file staging.

In [11]:
%%q
// Objects payload imported to 'instruments' table
data:((`instrumentid`sym`category`decimals`pipdecimals)!(77;"USDBRL";"EM";4;4);(`instrumentid`sym`category`decimals`pipdecimals)!(78;"USDKRW";"EM";2;2));
job:session.importData[`table`data!(`instruments;data)]

In [12]:
%%q
// Rows payload imported to the 'fxquote' table
data:(("2026-01-21";"2026-01-21T10:00:00.000";"EURUSD";901.2;901.3);("2026-01-21";"2026-01-21T10:00:00.000";"EURUSD";901.2;901.3));
job:session.importData[`table`data`columnNames!(`fxquote;data;(`trddate`ts`sym`bid`ask))]

// Note: for rows payload, columnNames are required.

## Querying Tables
Run structured, SQL, or q queries against DB Service.

In [13]:
%%q
// Structured query
session.querySimple[`table`startTS`endTS`sortCols`limit!(`fxquote;2026.03.02D;2026.03.03D;enlist "ts"; 5)]

trddate    ts                            sym    bid     ask    
---------------------------------------------------------------
2026.03.02 2026.03.02D00:00:00.000000000 AUDUSD 0.67091 0.67094
2026.03.02 2026.03.02D00:00:00.000000000 EURUSD 1.16397 1.16399
2026.03.02 2026.03.02D00:00:00.000000000 GBPUSD 1.3419  1.34194
2026.03.02 2026.03.02D00:00:00.000000000 USDCAD 1.38744 1.3875 
2026.03.02 2026.03.02D00:00:00.000000000 USDJPY 158.162 158.167


In [14]:
%%q
// SQL query
session.querySQL[enlist[`query]!enlist "SELECT * FROM instruments WHERE category LIKE 'EM'"]

instrumentid sym      category decimals pipdecimals
---------------------------------------------------
14           "CHFZAR" "EM"     5        4          
29           "EURTRY" "EM"     5        4          
31           "EURZAR" "EM"     5        4          
41           "GBPZAR" "EM"     5        4          
61           "USDCNH" "EM"     5        4          
66           "USDINR" "EM"     5        4          
68           "USDMXN" "EM"     5        4          
73           "USDTHB" "EM"     3        2          
74           "USDTRY" "EM"     5        4          
75           "USDZAR" "EM"     5        4          
76           "ZARJPY" "EM"     3        2          
77           "USDBRL" "EM"     4        4          
78           "USDKRW" "EM"     2        2          


In [15]:
%%q
// QSQL query
session.queryQ[enlist[`query]!enlist "select o:first bid,h:max bid,l:min bid,c:last bid by trddate,sym from fxquote"]

trddate    sym   | o       h       l       c      
-----------------| -------------------------------
2026.01.21 EURUSD| 901.2   901.2   901.2   901.2  
2026.03.02 AUDUSD| 0.67091 0.67469 0.67065 0.67307
2026.03.02 EURUSD| 1.16397 1.1768  1.16325 1.17268
2026.03.02 GBPUSD| 1.3419  1.34913 1.34101 1.34404
2026.03.02 USDCAD| 1.38744 1.3879  1.3814  1.38336
2026.03.02 USDJPY| 158.162 158.602 157.476 158.151
2026.03.03 AUDUSD| 0.6731  0.67781 0.67273 0.6754 
2026.03.03 EURUSD| 1.17258 1.1743  1.16703 1.16726
2026.03.03 GBPUSD| 1.34401 1.34588 1.33996 1.34176
2026.03.03 USDCAD| 1.38338 1.38441 1.37855 1.3844 
2026.03.03 USDJPY| 158.177 158.529 157.746 158.461


## Deleting tables

In [16]:
%%q
// List tables (expected: 'fxquote' and 'instruments')
session.listTables[]

"fxquote"
"instruments"


In [17]:
%%q
// Drop the 'instruments' table
session.dropTable["instruments"]

name    | "e4a813aa-441f-cff2-a38b-d766863d27a7"
pipeline| ""
database| "db"
updtype | "schemaChange"
status  | "completed"
details | ()
tbls    | ()
dates   | ()
progress| `cmdCurrent`cmdIndex`cmdTotal`subCurrent`subIndex`subTotal!("";0f;0..
error   | ""
warnings| ()
updated | "2026-04-27T12:24:02.684141649"


In [18]:
%%q
// List tables (expected: instruments is gone)
session.listTables[]

"fxquote"
